# 2D hydrogen atom and hydrogenic impurity

### Analytic solution + finite elements from scratch

This notebook documents the two ingredients of the project:

1. **Analytic solution** of the 2D Coulomb problem (the thesis appendix B).
2. **In-house FEM solver** (P2 elements, Delaunay mesh, no FEM library) that
   reproduces the analytic levels for the vacuum atom and for a donor in GaAs.

The same dimensionless equation describes both physical systems; only the
effective Bohr radius `a*` and the effective Rydberg `Ry*` change.

**Roadmap**

1. Material parameters and the two length/energy scales.
2. Analytic energies for `n = 0 ... 7`.
3. Analytic orbitals for vacuum and GaAs.
4. The derivation (separation of variables).
5. FEM architecture and validation results.
6. How to regenerate everything with `scripts/validar.py`.


## Step 1 — imports and materials

`VACIO` is the hydrogen atom in vacuum (`m* = m0`, `eps_r = 1`).
`GAAS` is a hydrogenic donor in GaAs (`m* = 0.067 m0`, `eps_r = 12.9`).


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")

import atomo2d.analitica as an
import atomo2d.fem as fem
from atomo2d.fem import GAAS, VACIO
from atomo2d.figuras import plot_analytic_orbitals

for mat in (VACIO, GAAS):
    print(f"{mat.name:7s}  a* = {mat.a_star_nm:10.6f} nm   Ry* = {mat.ry_star_mev:10.6f} meV")


## Step 2 — analytic energies

The 2D hydrogenic levels are

$$
E_n = -\frac{Ry^*}{\left(n+\tfrac12\right)^2},
\qquad n = 0,1,2,\dots
$$

For the vacuum atom `Ry* = 13.606 eV`; for GaAs `Ry* = 5.478 meV`.
The ground state is `-4 Ry*` in both cases, because the problem is the same
in dimensionless units.


In [ ]:
tabla = pd.DataFrame({
    "n": np.arange(8),
    "vacuum (meV)": [an.energia(n, VACIO) for n in range(8)],
    "GaAs (meV)":  [an.energia(n, GAAS) for n in range(8)],
})
tabla


## Step 3 — analytic orbitals

The radial functions are

$$
R_{nm}(r) = N_{nm}\,\rho^{|m|}e^{-\rho/2}
L_{n-|m|}^{2|m|}(\rho),
\qquad
\rho = \frac{2r}{(n+\tfrac12)a^*},
$$

with `|m| <= n`. The normalization `N_nm` is fixed by
`\int_0^\infty |R_{nm}|^2 r dr = 1` and computed by quadrature in
`atomo2d.analitica`.

The real orbitals used below are:

- `s`: `m = 0`, angular part `1/sqrt(2 pi)`.
- `p_x`, `p_y`: `m = 1`, angular parts `cos(phi)/sqrt(pi)` and `sin(phi)/sqrt(pi)`.
- `d_xy`: `m = 2`, angular part `sin(2 phi)/sqrt(pi)`.

The vacuum orbitals live on the scale of `a* = 0.053 nm`; the GaAs orbitals
on the scale of `a* = 10.19 nm`.


In [ ]:
fig_vac = plot_analytic_orbitals(
    VACIO, an.ESTADOS, r_zoom_nm=0.8,
    title="2D hydrogen atom in vacuum (analytic, a* = 0.0529 nm)",
)
plt.show()


In [ ]:
fig_gaas = plot_analytic_orbitals(
    GAAS, an.ESTADOS, r_zoom_nm=100.0,
    title="Hydrogenic donor in GaAs (analytic, a* = 10.19 nm)",
)
plt.show()


## Step 4 — derivation (thesis appendix B)

The Hamiltonian in polar coordinates is

$$
\hat H = -\frac{\hbar^2}{2\mu}
\left(\frac{\partial^2}{\partial r^2}
+\frac1r\frac{\partial}{\partial r}
+\frac1{r^2}\frac{\partial^2}{\partial \phi^2}\right)
-\frac{e^2}{4\pi\varepsilon_0 r}.
$$

Separation of variables, `psi(r, phi) = R_nm(r) Phi_m(phi)`, gives

$$
\Phi_m(\phi) = \frac{1}{\sqrt{2\pi}}e^{im\phi},
\qquad m = 0, \pm1, \dots, \pm n,
$$

and the radial equation whose solution is the `R_nm` above. The energies
follow from the boundary condition at infinity:

$$
E_n = -\frac{Ry}{(n+\tfrac12)^2}
\quad\text{(vacuum)},
\qquad
E_n = -\frac{Ry^*}{(n+\tfrac12)^2}
\quad\text{(semiconductor)}.
$$

The original Spanish derivation is reproduced verbatim in
[`docs/analitica.md`](../docs/analitica.md).


## Step 5 — FEM solver architecture

`src/atomo2d/fem.py` contains a complete P2 finite-element solver:

| Block | What it does |
|---|---|
| `construir_malla` | Delaunay mesh with geometric refinement towards the impurity; Dirichlet boundary at `R`. |
| `p2_shape` / `p2_shape_array` | Six quadratic Lagrange shape functions and their gradients. |
| `cuadratura_7` | Seven-point Dunavant quadrature of degree 5. |
| `ensamblar` | Sparse `K`, `M` and angular-momentum matrices; Duffy-type treatment of triangles touching `r = 0`. |
| `resolver_modos` | Shift-invert `eigsh` for `K u = E M u`. |
| `evaluador_p2` | Evaluates `psi_h` at arbitrary points inside the mesh. |
| `descomponer_angular` | Diagonalizes `L_z` in each degenerate subspace to label `m`. |

The solver is **dimensionless**: `x -> x/a*`, `E -> E/Ry*`, and
`H -> -laplacian - 2/r`. The material enters only through `a*` and `Ry*`.


In [ ]:
import inspect
print(inspect.signature(fem.construir_malla))
print()
print(inspect.signature(fem.ensamblar))
print()
print("7-point quadrature:")
xi, eta, w = fem.cuadratura_7()
print("sum(w) =", w.sum(), " (reference triangle area = 0.5)")


## Step 6 — FEM results

The script `scripts/validar.py` solves both cases at two domain sizes and
writes `results/resultados.json`, `results/energies.csv`, and the orbital and
mesh figures. The table below is the validation against the analytic
energies.


In [ ]:
import json
from IPython.display import Image, display

res = json.load(open("../results/resultados.json"))

for case in ("vacuum", "gaas"):
    datos = res[case]
    print(f"\n=== {datos['title']} ===")
    print(f"R = {datos['mesh_R']['R_nm']:.4g} nm, "
          f"{datos['mesh_R']['nodes']} nodes, "
          f"{datos['mesh_R']['ndof']} dof")
    tabla = pd.DataFrame(
        datos["table"],
        columns=["state", "(n,m)", "E_theory", "E_FEM_R", "diff_R",
                 "E_FEM_Rc", "diff_Rc"],
    )
    display(tabla)


### FEM orbitals — GaAs donor

The orbitals below come from the FEM solver, not from the analytic formula.
They are shown as they come out of the eigensolver.


In [ ]:
display(Image(filename="../figures/orbitals_gaas.png", width=1100))


### FEM orbitals — vacuum atom

The same solver, with `a* = 0.0529 nm` and a domain of `40 a* = 2.12 nm`.


In [ ]:
display(Image(filename="../figures/orbitals_vacuum.png", width=1100))


### Mesh

The mesh is refined geometrically towards the origin, where the Coulomb
potential is singular.


In [ ]:
display(Image(filename="../figures/mesh_gaas.png", width=520))
display(Image(filename="../figures/mesh_vacuum.png", width=520))


## Step 7 — regenerate everything

```bash
python scripts/validar.py
```

The script takes a few minutes on a laptop. It writes:

- `results/resultados.json`
- `results/energies.csv`
- `figures/orbitals_gaas.png`
- `figures/orbitals_vacuum.png`
- `figures/mesh_gaas.png`
- `figures/mesh_vacuum.png`

## Step 8 — what to remember

- The **analytic solution** is exact and sets the reference.
- The **FEM solver is from scratch**: Delaunay mesh, P2 elements, 7-point
  quadrature, sparse assembly, shift-invert eigensolver.
- GaAs: the eight levels agree with the analytic values to **≤ 0.009 meV**.
- Vacuum: the relative error is **~5 × 10^-5** (a few meV on a 54 eV scale);
  the orbitals have the correct nodal structure.
- The same code is the validation engine of the quantum dot chain thesis.
